# 🧠 MLP vs. Baselines — Comparação de Modelos
> **Projeto:** Telco Customer Churn Prediction  
> **Branch:** `feat/register_mlflow_model`  
> **Objetivo:** Comparar um MLP (rede neural) contra modelos lineares e baseados em árvore, usando ≥ 4 métricas de avaliação, com rastreamento via MLflow.

**Modelos avaliados:**
| Família | Modelos |
|---------|---------|
| Lineares | Logistic Regression, Ridge Classifier |
| Árvores | Decision Tree, Random Forest, CatBoost |
| Rede Neural | MLP (PyTorch) |

**Métricas avaliadas:** Accuracy, F1-score (macro), ROC-AUC, Precision, Recall


## 1. Importações e Configuração

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

import mlflow
import mlflow.sklearn
import mlflow.pytorch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print("✅ Imports OK")
print(f"   PyTorch  : {torch.__version__}")
print(f"   MLflow   : {mlflow.__version__}")


## 2. Configuração do MLflow

In [ ]:
# Aponta para o banco SQLite já existente no projeto
MLFLOW_TRACKING_URI = "sqlite:///./mlruns.db"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

EXPERIMENT_NAME = "MLP_vs_Baselines_Churn"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"📁 Tracking URI : {mlflow.get_tracking_uri()}")
print(f"🧪 Experimento  : {EXPERIMENT_NAME}")


## 3. Carregamento e Pré-processamento dos Dados

In [ ]:
# ──────────────────────────────────────────────────
# Adapte o caminho se o dataset estiver em src/data/
# ──────────────────────────────────────────────────
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head(3)


In [ ]:
# ── Limpeza básica
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)
df.drop(columns=["customerID"], inplace=True, errors="ignore")

# ── Encode alvo
df["Churn"] = (df["Churn"] == "Yes").astype(int)

# ── One-hot para categóricas
cat_cols = df.select_dtypes(include="object").columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"Shape pós-encoding: {df.shape}")
print(f"Distribuição Churn:\n{df['Churn'].value_counts()}")


In [ ]:
TARGET = "Churn"
X = df.drop(columns=[TARGET]).values.astype(np.float32)
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Treino: {X_train_sc.shape} | Teste: {X_test_sc.shape}")


## 4. Funções Auxiliares

In [ ]:
def compute_metrics(name, y_true, y_pred, y_prob=None):
    """Calcula e retorna dicionário com ≥ 4 métricas.""""
    metrics = {
        "model"     : name,
        "accuracy"  : round(accuracy_score(y_true, y_pred), 4),
        "f1_macro"  : round(f1_score(y_true, y_pred, average="macro"), 4),
        "precision" : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall"    : round(recall_score(y_true, y_pred), 4),
    }
    if y_prob is not None:
        metrics["roc_auc"] = round(roc_auc_score(y_true, y_prob), 4)
    else:
        metrics["roc_auc"] = None
    return metrics


def log_and_evaluate(name, model, X_tr, y_tr, X_te, y_te, extra_params=None):
    """Treina, avalia e loga um modelo sklearn no MLflow.""""
    with mlflow.start_run(run_name=name):
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        # Probabilidade para ROC-AUC (se suportado)
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_te)[:, 1]
        elif hasattr(model, "decision_function"):
            y_prob = model.decision_function(X_te)
        else:
            y_prob = None

        m = compute_metrics(name, y_te, y_pred, y_prob)

        # Loga parâmetros e métricas no MLflow
        if extra_params:
            mlflow.log_params(extra_params)
        mlflow.log_metric("accuracy",  m["accuracy"])
        mlflow.log_metric("f1_macro",  m["f1_macro"])
        mlflow.log_metric("precision", m["precision"])
        mlflow.log_metric("recall",    m["recall"])
        if m["roc_auc"]:
            mlflow.log_metric("roc_auc", m["roc_auc"])
        mlflow.sklearn.log_model(model, artifact_path="model")

    return m

results = []
print("✅ Funções auxiliares definidas.")


## 5. Modelos Lineares

In [ ]:
# ── 5.1 Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
m = log_and_evaluate(
    "LogisticRegression", lr,
    X_train_sc, y_train, X_test_sc, y_test,
    extra_params={"C": 1.0, "max_iter": 1000}
)
results.append(m)
print(f"LR  → Acc={m['accuracy']} | F1={m['f1_macro']} | AUC={m['roc_auc']}")


In [ ]:
# ── 5.2 Ridge Classifier
ridge = RidgeClassifier(alpha=1.0)
m = log_and_evaluate(
    "RidgeClassifier", ridge,
    X_train_sc, y_train, X_test_sc, y_test,
    extra_params={"alpha": 1.0}
)
results.append(m)
print(f"Ridge → Acc={m['accuracy']} | F1={m['f1_macro']}")


## 6. Modelos Baseados em Árvore

In [ ]:
# ── 6.1 Decision Tree
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
m = log_and_evaluate(
    "DecisionTree", dt,
    X_train_sc, y_train, X_test_sc, y_test,
    extra_params={"max_depth": 6}
)
results.append(m)
print(f"DT  → Acc={m['accuracy']} | F1={m['f1_macro']} | AUC={m['roc_auc']}")


In [ ]:
# ── 6.2 Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
m = log_and_evaluate(
    "RandomForest", rf,
    X_train_sc, y_train, X_test_sc, y_test,
    extra_params={"n_estimators": 200, "max_depth": 8}
)
results.append(m)
print(f"RF  → Acc={m['accuracy']} | F1={m['f1_macro']} | AUC={m['roc_auc']}")


In [ ]:
# ── 6.3 CatBoost (instalado via poetry no projeto)
try:
    from catboost import CatBoostClassifier
    cb = CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05,
        eval_metric="AUC", random_seed=42, verbose=0
    )
    m = log_and_evaluate(
        "CatBoost", cb,
        X_train_sc, y_train, X_test_sc, y_test,
        extra_params={"iterations": 300, "depth": 6, "lr": 0.05}
    )
    results.append(m)
    print(f"CB  → Acc={m['accuracy']} | F1={m['f1_macro']} | AUC={m['roc_auc']}")
except ImportError:
    print("⚠️  CatBoost não encontrado — pulando.")


## 7. MLP — Rede Neural Multicamadas

### 7.1 MLP sklearn (rápido, baseline neural)

In [ ]:
mlp_sklearn = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    max_iter=200,
    learning_rate_init=1e-3,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
m = log_and_evaluate(
    "MLP_sklearn", mlp_sklearn,
    X_train_sc, y_train, X_test_sc, y_test,
    extra_params={"hidden_layers": "128-64-32", "activation": "relu",
                  "solver": "adam", "max_iter": 200}
)
results.append(m)
print(f"MLP (sklearn) → Acc={m['accuracy']} | F1={m['f1_macro']} | AUC={m['roc_auc']}")


### 7.2 MLP PyTorch (versão completa com épocas rastreadas)

In [ ]:
class ChurnMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


In [ ]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS     = 50
BATCH_SIZE = 256
LR         = 1e-3

X_tr_t = torch.tensor(X_train_sc, dtype=torch.float32)
y_tr_t = torch.tensor(y_train,    dtype=torch.float32)
X_te_t = torch.tensor(X_test_sc,  dtype=torch.float32)
y_te_t = torch.tensor(y_test,     dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_tr_t, y_tr_t),
    batch_size=BATCH_SIZE, shuffle=True
)

model_pt   = ChurnMLP(X_train_sc.shape[1]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss()
optimizer  = optim.Adam(model_pt.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

train_losses = []

with mlflow.start_run(run_name="MLP_PyTorch"):
    mlflow.log_params({
        "architecture": "128-BN-ReLU-Drop0.3 | 64-BN-ReLU-Drop0.2 | 32-ReLU | 1",
        "optimizer": "Adam", "lr": LR, "epochs": EPOCHS,
        "batch_size": BATCH_SIZE, "weight_decay": 1e-4
    })

    for epoch in range(1, EPOCHS + 1):
        model_pt.train()
        epoch_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model_pt(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        mlflow.log_metric("train_loss", avg_loss, step=epoch)

        if epoch % 10 == 0:
            print(f"  Época {epoch:>3}/{EPOCHS} | loss={avg_loss:.4f}")

    # ── Avaliação
    model_pt.eval()
    with torch.no_grad():
        logits  = model_pt(X_te_t.to(DEVICE)).cpu()
        y_prob  = torch.sigmoid(logits).numpy()
        y_pred  = (y_prob >= 0.5).astype(int)

    m_pt = compute_metrics("MLP_PyTorch", y_test, y_pred, y_prob)
    for k, v in m_pt.items():
        if k != "model" and v is not None:
            mlflow.log_metric(k, v)

    mlflow.pytorch.log_model(model_pt, artifact_path="model")

results.append(m_pt)
print(f"\nMLP (PyTorch) → Acc={m_pt['accuracy']} | F1={m_pt['f1_macro']} | AUC={m_pt['roc_auc']}")


## 8. Tabela Comparativa de Métricas

In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results.set_index("model").sort_values("roc_auc", ascending=False)

# Destaca melhor valor de cada coluna
def highlight_max(s):
    is_max = s == s.max()
    return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_max]

print("\n📊 Comparação Final:\n")
print(df_results.to_string())

df_results.style.apply(highlight_max, subset=["accuracy","f1_macro","precision","recall","roc_auc"])


## 9. Visualizações

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
metrics_to_plot = ["accuracy", "f1_macro", "roc_auc"]
df_plot = df_results[metrics_to_plot].reset_index()
df_melt = df_plot.melt(id_vars="model", var_name="Métrica", value_name="Score")

# ── Barplot métricas
sns.barplot(
    data=df_melt, x="model", y="Score", hue="Métrica",
    palette="Set2", ax=axes[0]
)
axes[0].set_title("Comparação de Métricas por Modelo", fontsize=13)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha="right")
axes[0].set_ylim(0.5, 1.0)
axes[0].legend(loc="lower right")
axes[0].set_xlabel("")

# ── Curva de loss do PyTorch
axes[1].plot(range(1, len(train_losses)+1), train_losses, color="steelblue", linewidth=2)
axes[1].set_title("Curva de Loss — MLP PyTorch", fontsize=13)
axes[1].set_xlabel("Época")
axes[1].set_ylabel("BCEWithLogitsLoss")
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig("../docs/mlp_vs_baselines_metricas.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Gráfico salvo em src/docs/")


In [ ]:
# ── Matriz de confusão — MLP PyTorch
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Não Churn", "Churn"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Matriz de Confusão — MLP PyTorch", fontsize=12)
plt.tight_layout()
plt.show()


## 10. Relatório Final e Conclusão

In [ ]:
print("=" * 60)
print("  RELATÓRIO FINAL — MLP vs. Baselines")
print("=" * 60)

best_auc   = df_results["roc_auc"].idxmax()
best_f1    = df_results["f1_macro"].idxmax()
best_acc   = df_results["accuracy"].idxmax()
mlp_auc    = df_results.loc["MLP_PyTorch", "roc_auc"] if "MLP_PyTorch" in df_results.index else None

print(f"\n🏆 Melhor ROC-AUC   : {best_auc} ({df_results['roc_auc'].max():.4f})")
print(f"🏆 Melhor F1 Macro  : {best_f1}  ({df_results['f1_macro'].max():.4f})")
print(f"🏆 Melhor Accuracy  : {best_acc} ({df_results['accuracy'].max():.4f})")
if mlp_auc:
    print(f"\n🧠 MLP PyTorch AUC  : {mlp_auc:.4f}")
    diff = mlp_auc - df_results.drop("MLP_PyTorch", errors="ignore")["roc_auc"].max()
    status = "✅ MLP supera" if diff > 0 else "⚠️  MLP abaixo de"
    print(f"   {status} melhor baseline por {abs(diff):.4f} de AUC")

print("\n" + "=" * 60)
print("  Experimentos registrados em MLflow:")
print(f"  sqlite:///./mlruns.db → Experimento '{EXPERIMENT_NAME}'")
print("=" * 60)


## 11. Como Visualizar no MLflow UI

```bash
# Na raiz do projeto (onde está mlruns.db)
cd src
mlflow ui --backend-store-uri sqlite:///mlruns.db --port 5000
```
Abra `http://localhost:5000` e acesse o experimento **MLP_vs_Baselines_Churn**.
